# 🚀 Advanced PySpark Performance Tuning & Optimization (Part 3)
This notebook covers the final set of production optimization techniques: Shuffle Partition Tuning Calculations, Partition Pruning & Directory Layout Design, and Dynamic Allocation & Memory Fraction Tuning.

--- 
## 10. Dynamic Shuffle Partition Sizing Math
### **Concept & Explanation**
* **The Bottleneck:** The default shuffle partition count (`spark.sql.shuffle.partitions = 200`) is a common performance trap. For small datasets, 200 partitions cause excessive task scheduling overhead and tiny output files. For multi-terabyte datasets, 200 partitions cause executor OOM errors due to oversized shuffle blocks.
* **The Formula:** The ideal shuffle partition count targets ~128MB to 200MB per partition block.
* **Production Best Practice:** Calculate shuffle partitions dynamically based on your dataset's compressed shuffle size before heavy wide transformations.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName(
        "Optimization_10_Shuffle_Partition_Math"
    ) \
    .master("local[*]") \
    .getOrCreate()

# Step 1: Simulate estimating dataset size and calculating optimal partitions
TARGET_PARTITION_SIZE_MB = 128
estimated_shuffle_data_gb = (
    15.0  # Example: 15 GB intermediate shuffle data
)
estimated_shuffle_data_mb = (
    estimated_shuffle_data_gb * 1024
)

calculated_partitions = int(
    estimated_shuffle_data_mb // TARGET_PARTITION_SIZE_MB
)
# Ensure at least a reasonable minimum
optimal_partitions = max(
    calculated_partitions, 50
)

print(
    f"Target Shuffle Partitions Set to: {optimal_partitions}"
)
spark.conf.set(
    "spark.sql.shuffle.partitions", str(optimal_partitions)
)

# Step 2: Run a wide transformation with the tuned partition count
df = spark.range(0, 1000000).withColumn(
    "group_key", F.col("id") % 500
)
agg_df = df.groupBy("group_key").count()
print("=== GroupBy Plan with Dynamic Partition Count ===")
agg_df.explain(True)

--- 
## 11. Partition Pruning via Directory Layouts
### **Concept & Explanation**
* **The Bottleneck:** Scanning an entire data lake directory without partition filters forces Spark to read every file footer across all dates or regions.
* **How Partition Pruning Works:** Writing datasets using `.partitionBy("year", "month")` organizes data into a hierarchical folder structure (`/year=2026/month=03/`). When a query includes a filter on `year` and `month`, Spark completely skips scanning entire folder paths at the driver level before task generation.
* **Production Best Practice:** Always partition high-frequency analytical tables by low-cardinality time dimensions (date, year, month).

In [ ]:
import os
import shutil

output_dir = "/tmp/partition_pruning_demo"
if os.path.exists(output_dir):
  shutil.rmtree(output_dir)

# Step 1: Create dataset with partition columns
df_base = spark.range(0, 200000).withColumn(
    "year",
    F.when(F.col("id") % 2 == 0, 2025).otherwise(2026),
)

df_base.write.partitionBy("year").mode("overwrite").parquet(
    output_dir
)

# Step 2: Read back with partition filter
df_pruned = spark.read.parquet(output_dir).filter(
    "year = 2026"
)
print(
    "=== Physical Plan Showing PartitionFilters Pruning ==="
)
df_pruned.explain(True)

--- 
## 12. Executor Memory Fraction Tuning (`spark.memory.fraction`)
### **Concept & Explanation**
* **Memory Architecture:** Spark divides executor memory into Storage (caching) and Execution (shuffles, joins, aggregations) managed by a unified region defined by `spark.memory.fraction` (default 0.6 or 60%).
* **The Tuning Strategy:** If your workloads involve heavy shuffles and large joins rather than heavy dataframe caching, increasing execution headroom prevents premature disk spilling.
* **Production Best Practice:** Tune memory fractions carefully in cluster configuration templates (`spark-defaults.conf`) based on profiling data from the Spark UI Storage and Stages tabs.

In [ ]:
# Step 1: Inspect active memory configurations
print(
    "Spark Memory Fraction:",
    spark.conf.get("spark.memory.fraction", "Default (0.6)"),
)
print(
    "Spark Memory Storage Fraction:",
    spark.conf.get(
        "spark.memory.storageFraction", "Default (0.5)"
    ),
)

# Step 2: Example configuration snippet for cluster initialization
print(
    "\nRecommended production config template for shuffle-heavy workloads:"
)
print(
    'spark.conf.set("spark.memory.fraction", "0.8")'
)  # Allocate 80% of heap to unified memory
print(
    'spark.conf.set("spark.memory.storageFraction", "0.3")'
)  # Reserve 30% for caching, leaving 70% for execution/joins

spark.stop()